# SOFR butterfly grid — what the option strikes say the distribution is

Rebuilds the 12bp call-fly grid from listed settles.

> *"12bp wide, 6bp body increments, prev day settle. Buy lower + Sell 2× body +
> Buy upper (calls). Wings = ±12bp. Max payout = 0.125. Imp Prob = Fly Settle ÷
> 0.125. Yield = 100 − body."*

Listed SOFR strikes are 6.25bp apart and the wings sit two steps out, so the fly
pays 0.125 at the body and tapers linearly to zero ±12.5bp away.

## Two traps, both of which change the answer

1. **Barchart serves no EOD for deep-ITM calls.** A naive rebuild stops dead at
   the first body needing a call struck below the future — which is exactly where
   the pin is. Fill via put-call parity `C(K) = P(K) + DF·(F − K)` from the OTM
   puts, the same route the Breeden–Litzenberger pipeline uses.
2. **"Imp Prob" is not a probability.** Adjacent triangular kernels of half-width
   12.5bp on a 6.25bp body grid overlap ~2×, so the column **sums to 2.0**. Halve
   it for a first-order true mass, and never add it across bodies.

## What it is for

The forward is the *mean* of the distribution. The grid shows the *mode*. When
they are far apart the priced path is not the likely path, and the gap is the
tail doing the work.

In [1]:
import os

os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")

import dataclasses
import datetime
import math
import pathlib
import pickle
import sys
import time

import numpy as np
import pandas as pd

_here = pathlib.Path.cwd()
_REPO = next(p for p in [_here, *_here.parents] if (p / "RVUtils").is_dir())
sys.path.insert(0, str(_REPO))

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 120)
print("repo:", _REPO)

repo: C:\Users\chris\clee\ARBS-fly


## 1. CONFIG

In [2]:
@dataclasses.dataclass(frozen=True)
class GridConfig:
    as_of: datetime.date = datetime.date(2026, 8, 21)
    """Settlement date for the option marks. The grid is a prev-day-settle
    object; using an intraday mark mixes vintages across strikes."""
    contracts: tuple = ("SFRU26", "SFRV26", "SFRX26", "SFRZ26")
    """Option expiries. Two-digit year — ``SFRU6`` raises. V6/X6 are serial
    monthlies and settle on the SFRZ26 future, which is why their forward is not
    their own name."""
    source: str = "BARCHART_STIRFO-QL"
    wing: float = 0.125
    """Wing distance in price points: two listed strike steps of 6.25bp."""
    discount_factor: float = 0.998
    """For the parity fill. The term is second order at these expiries; a 20bp
    error in DF moves a fly by well under a tick."""
    body_hi: float = 96.75
    """Bodies above this are all noise for a future near 96; shown separately."""


GCFG = GridConfig()
CACHE = _REPO / "docs" / "curvefly" / f"sfr_smiles_{GCFG.as_of:%Y%m%d}.pkl"
GCFG

GridConfig(as_of=datetime.date(2026, 8, 21), contracts=('SFRU26', 'SFRV26', 'SFRX26', 'SFRZ26'), source='BARCHART_STIRFO-QL', wing=0.125, discount_factor=0.998, body_hi=96.75)

## 2. Fetch the smiles

`fetch_sabr_smile` returns a `STIRFutureOptionSABRSmile`, **not** a DataFrame —
`.points` carry `strike_price`, `right`, `market_price`, `open_interest`. It is
one HTTP call per strike with no cached history, so this cell caches to disk;
re-running is free. Expect noisy "no columns to parse" errors for deep-OTM and
deep-ITM strikes — those are absent quotes, not failures.

In [3]:
if CACHE.exists():
    points = pickle.loads(CACHE.read_bytes())
    print("loaded cache:", {k: len(v) for k, v in points.items()})
else:
    from MDP.STIRFutures.STIRFutureOptionMDP import STIRFutureOptionMDP

    mdp = STIRFutureOptionMDP(source=GCFG.source)
    points = {}
    for c in GCFG.contracts:
        sm = mdp.fetch_sabr_smile({"symbol": c, "as_of": GCFG.as_of,
                                   "strike_offsets_bps": "listed"})
        points[c] = [dict(strike=p.strike_price, right=p.right,
                          px=p.market_price, oi=p.open_interest) for p in sm.points]
        print(f"{c}: {len(points[c])} points on {sm.underlying_contract}")
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    CACHE.write_bytes(pickle.dumps(points))

loaded cache: {'SFRU26': 53, 'SFRV26': 52, 'SFRX26': 51, 'SFRZ26': 54}


## 3. Underlying forwards, then the parity fill

The forward matters twice: it decides which strikes are ITM (and so need
synthesising) and it is the mean the mode gets compared against. Serial monthlies
take the **quarterly's** future, not one of their own.

In [4]:
from BT.serff.futures_data import backfill_settles, settle_panel

hist = backfill_settles(datetime.date(2026, 5, 1), GCFG.as_of,
                        symbols=["SR3U26", "SR3Z26"], show_progress=False)
px_panel = settle_panel(hist)
row = px_panel.loc[pd.Timestamp(GCFG.as_of)]
FUT = {"SFRU26": float(row["SR3U26"]), "SFRV26": float(row["SR3Z26"]),
       "SFRX26": float(row["SR3Z26"]), "SFRZ26": float(row["SR3Z26"])}
print({k: round(v, 4) for k, v in FUT.items()})


def build_grid(pts, F, wing=GCFG.wing, df_=GCFG.discount_factor):
    d = pd.DataFrame(pts).dropna(subset=["px"])
    calls = d[d.right.str.upper().str.startswith("C")].groupby("strike")["px"].last()
    puts = d[d.right.str.upper().str.startswith("P")].groupby("strike")["px"].last()
    synth = {float(K): float(p) + df_ * (F - K) for K, p in puts.items() if K < F}
    merged = dict(synth)
    merged.update({float(k): float(v) for k, v in calls.items()})   # real calls win
    s = pd.Series(merged).sort_index()
    rows = []
    for K in s.index:
        lo, hi = round(K - wing, 4), round(K + wing, 4)
        if lo in s.index and hi in s.index:
            fly = float(s[lo] - 2 * s[K] + s[hi])
            rows.append(dict(body=K, fly=fly, imp=fly / wing,
                             src="parity" if (K in synth and K not in calls.index) else "call"))
    return pd.DataFrame(rows).set_index("body")


grids = {c: build_grid(points[c], FUT[c]) for c in points}
for c, g in grids.items():
    print(f"{c}: {len(g)} bodies | column sums to {g.imp.sum():.2f} "
          f"(must be ~2.0 -- see trap 2)")

{'SFRU26': 96.2, 'SFRV26': 96.045, 'SFRX26': 96.045, 'SFRZ26': 96.045}
SFRU26: 45 bodies | column sums to 2.00 (must be ~2.0 -- see trap 2)
SFRV26: 45 bodies | column sums to 1.98 (must be ~2.0 -- see trap 2)
SFRX26: 45 bodies | column sums to 2.00 (must be ~2.0 -- see trap 2)
SFRZ26: 45 bodies | column sums to 1.96 (must be ~2.0 -- see trap 2)


## 4. GATE — against a published grid

The external check. JWS Macro #8 printed the SFRU6 column off the same
prev-day settle; these are his numbers. Pure-call bodies should match to about a
tick; parity-filled bodies carry the residual, which is where any DF or
settle-vintage error shows up.

In [5]:
PUBLISHED = {96.0000: 12.0, 96.0625: 16.0, 96.1250: 18.0, 96.1875: 36.0,
             96.2500: 52.0, 96.3125: 38.0, 96.3750: 16.0, 96.4375: 4.0}
g = grids["SFRU26"]
errs = []
for K, want in sorted(PUBLISHED.items()):
    got = float(g.imp.get(K, np.nan)) * 100
    src = g.src.get(K, "-")
    if np.isfinite(got):
        errs.append(abs(got - want))
    print(f"  body {K:.4f} [{src:>6}]  published {want:>5.1f}%   "
          f"rebuilt {got:>6.1f}%   diff {got-want:>+6.1f}")
e = np.array(errs)
print(f"\n  matched {len(e)}/{len(PUBLISHED)} | mean |diff| {e.mean():.1f}pp | "
      f"max {e.max():.1f}pp")
by_src = {s: [abs(float(g.imp.get(K, np.nan)) * 100 - w)
              for K, w in PUBLISHED.items() if g.src.get(K, "-") == s
              and np.isfinite(g.imp.get(K, np.nan))] for s in ("call", "parity")}
for s, v in by_src.items():
    if v:
        print(f"  {s:>6} bodies: mean {np.mean(v):.1f}pp over {len(v)}")

  body 96.0000 [parity]  published  12.0%   rebuilt   10.0%   diff   -2.0
  body 96.0625 [parity]  published  16.0%   rebuilt   20.0%   diff   +4.0
  body 96.1250 [parity]  published  18.0%   rebuilt   23.9%   diff   +5.9
  body 96.1875 [  call]  published  36.0%   rebuilt   31.8%   diff   -4.2
  body 96.2500 [  call]  published  52.0%   rebuilt   45.9%   diff   -6.1
  body 96.3125 [  call]  published  38.0%   rebuilt   38.0%   diff   +0.0
  body 96.3750 [  call]  published  16.0%   rebuilt   14.0%   diff   -2.0
  body 96.4375 [  call]  published   4.0%   rebuilt    4.0%   diff   -0.0

  matched 8/8 | mean |diff| 3.0pp | max 6.1pp
    call bodies: mean 2.5pp over 5
  parity bodies: mean 4.0pp over 3


## 5. The grid, and the reading that matters

The forward is the mean; the peak body is the mode. A large gap between them says
the distribution is skewed by a tail — and that the number everyone quotes (the
forward) is not the outcome the option market thinks is most likely.

In [6]:
view = pd.DataFrame({c: (gg.imp * 100).round(1) for c, gg in grids.items()})
view.insert(0, "yield_%", [round(100 - b, 2) for b in view.index])
near = view.loc[view.index <= GCFG.body_hi]
print(near.to_string())

print("\n=== mode vs forward ===")
for c, gg in grids.items():
    sub = gg[gg.index <= GCFG.body_hi]
    mode = float(sub.imp.idxmax())
    f = FUT[c]
    print(f"  {c}: future {f:.4f} ({100-f:.3f}%) | mode {mode:.4f} ({100-mode:.3f}%) "
          f"| gap {(100-mode)-(100-f):+.3f}% in RATE | peak {sub.imp.max()*100:.0f}% "
          f"(~{sub.imp.max()*50:.0f}% of true mass)")

         yield_%  SFRU26  SFRV26  SFRX26  SFRZ26
body                                            
94.6875     5.31     NaN     0.0     0.0    -2.0
94.7500     5.25     NaN     0.0     2.0     2.0
94.8125     5.19    -0.0    -0.0     2.0     2.0
94.8750     5.12     0.0     0.0     0.0    -2.0
94.9375     5.06     0.0     2.0     0.0     0.0
95.0000     5.00    -0.0     2.0    -2.0     2.0
95.0625     4.94     0.0    -2.0    -2.0    -0.0
95.1250     4.88     0.0    -2.0     2.0     2.0
95.1875     4.81    -0.0     0.0     2.0     2.0
95.2500     4.75    -0.0     2.0    -0.0    -2.0
95.3125     4.69     0.0     4.0     2.0    -2.0
95.3750     4.62     0.0     2.0     2.0     4.0
95.4375     4.56     0.0    -0.0     2.0     6.0
95.5000     4.50    -0.0     0.0     4.0     4.0
95.5625     4.44     0.0     2.0     2.0     4.0
95.6250     4.38     0.0     6.0     4.0     4.0
95.6875     4.31     2.0     8.0    10.0     8.0
95.7500     4.25     2.0    10.0    12.0    14.0
95.8125     4.19    

In [7]:
fig = go.Figure()
for c, gg in grids.items():
    sub = gg[(gg.index <= GCFG.body_hi) & (gg.index >= 95.5)]
    fig.add_trace(go.Scatter(x=[100 - b for b in sub.index], y=sub.imp * 100,
                             mode="lines+markers", name=c))
for c, f in FUT.items():
    if c == "SFRU26" or c == "SFRZ26":
        fig.add_vline(x=100 - f, line_dash="dot", line_width=1,
                      annotation_text=f"{c} fwd", annotation_position="top")
fig.update_layout(
    title="Butterfly grid — where the option market puts the mass, vs where the forward is",
    xaxis_title="body yield, %", yaxis_title="fly settle / 0.125, %  (sums to 2.0)",
    height=520, hovermode="x unified")
fig.show()

## 6. Halving it, and why that is the honest version

The column double-counts, so the peak is roughly half what it reads. The
normalised view below is the one to quote — and it still shows a sharp pin,
which is the point: the concentration is real, the headline number is not.

In [8]:
norm = pd.DataFrame({c: gg.imp / gg.imp.sum() for c, gg in grids.items()})
norm = norm.loc[(norm.index <= GCFG.body_hi) & (norm.index >= 95.5)]
norm.index = [round(100 - b, 3) for b in norm.index]
norm.index.name = "yield_%"
print("normalised so each column sums to 1.0 (%)")
print((norm * 100).round(1).to_string())
for c in norm.columns:
    top = norm[c].nlargest(3)
    print(f"  {c}: top-3 bodies hold {top.sum()*100:.0f}% of mass at yields "
          f"{', '.join(f'{i:.2f}%' for i in top.index)}")

normalised so each column sums to 1.0 (%)
         SFRU26  SFRV26  SFRX26  SFRZ26
yield_%                                
4.500      -0.0     0.0     2.0     2.0
4.438       0.0     1.0     1.0     2.0
4.375       0.0     3.0     2.0     2.0
4.312       1.0     4.0     5.0     4.1
4.250       1.0     5.1     6.0     7.2
4.188      -0.0     6.1     6.0     6.1
4.125       1.0     6.1     5.0     3.1
4.062       2.0     6.1     5.0     4.1
4.000       5.0     6.0     5.9     7.1
3.938      10.0     7.0     5.9     7.1
3.875      12.0    10.1     7.0     4.1
3.812      15.9    14.2     9.0     3.1
3.750      23.0    14.2    12.0    10.2
3.688      19.0     8.1    12.0    16.4
3.625       7.0     2.0     6.0    10.2
3.562       2.0     1.0     2.0     2.0
3.500       1.0     2.0     1.0     1.0
3.438       0.0     1.0    -1.0     1.0
3.375       0.0    -1.0     0.0    -1.0
3.312       0.0    -1.0     1.0    -0.0
3.250       0.0     1.0    -0.0     1.0
  SFRU26: top-3 bodies hold 58% of mas

## 7. Caveats

* The parity fill inherits any error in the forward and the discount factor. It
  is second order for these expiries but it is the reason the low bodies carry
  more residual against the published grid than the pure-call ones.
* A fly built from settles inherits settle staleness. Deep wings frequently have
  no trade and no meaningful settle; that is why several bodies print 0.0 or a
  small negative, which is quote noise rather than negative probability.
* Serial monthlies (`V6`, `X6`) settle on the quarterly future. Comparing their
  mode against their own name's forward is a category error.
* The triangular kernel understates a flat distribution and overstates a peaked
  one. That asymmetry is why the grid is good at *locating* a pin and bad at
  measuring how much is in it.